## Step 4 — SteamCharts Scraping & Panel Construction

Scrape monthly average and peak concurrent player counts from [steamcharts.com](https://steamcharts.com) for every game in `steam_matches_manual.json`, then assemble a long-format game–month panel and attach treatment metadata for the empirical analysis.

**Input:**  `../raw_data/steam_matches_manual.json` (Step 3 output) and `../raw_data/epic games data.csv` (for giveaway dates)
**Output:**

- `../raw_data/steamcharts/{appid}.json` — per-game cache (one file per game)
- `../raw_data/steamcharts/all_monthly.csv` — combined long-format panel; section 6 attaches `all_giveaway_dates` and `repeat_event_it`

The monthly table on each app page (`table.common-table`) is fully server-side rendered, so no JS is required. Re-running skips already-cached app IDs, so interruptions are safe.

**Bot detection:** SteamCharts is fronted by Cloudflare. The scraper uses a full Chrome 120 User-Agent with matching `Sec-Fetch-*` headers and a persistent session (1 second between requests).

Each per-game cache file looks like:
```json
{
  "appid": 264710,
  "steam_name": "Subnautica",
  "epic_name": "Subnautica",
  "months": [
    {"month": "2018-12", "avg_players": 4321.2, "peak_players": 8900},
    ...
  ]
}
```

## Sections
0. Diagnostic — verify we can reach SteamCharts
1. Build the scrape list from `steam_matches_manual.json`
2. Scrape SteamCharts
3. Inspect empty results
4. Build combined long-format CSV
5. Quick sanity check
6. Treatment variable construction


In [1]:
import json
import time
import re
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

DATA_DIR     = Path("../raw_data")
CACHE_DIR    = DATA_DIR / "steamcharts"
CACHE_DIR.mkdir(exist_ok=True)

MATCHES_FILE = DATA_DIR / "steam_matches_manual.json"
OUT_CSV      = CACHE_DIR / "all_monthly.csv"

RATE_LIMIT_S = 1.0   # seconds between requests
MAX_RETRIES  = 1

# Full Chrome 120 headers — needed to pass SteamCharts bot detection
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8"
    ),
    "Accept-Language":  "en-US,en;q=0.9",
    # Accept-Encoding intentionally omitted — letting requests/urllib3 handle
    # decompression automatically (setting it manually disables auto-decode)
    "Connection":       "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest":   "document",
    "Sec-Fetch-Mode":   "navigate",
    "Sec-Fetch-Site":   "same-origin",
    "Sec-Fetch-User":   "?1",
    "sec-ch-ua":        '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
}

# Persistent session shares cookies across requests (important for Cloudflare)
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

## 0. Diagnostic — verify we can reach SteamCharts

In [2]:
# Probe the HTML page for Subnautica and read the server-rendered monthly table.
# The table uses class="common-table" (not id="monthly-data") and contains
# real avg AND peak for every month — no JS rendering required.
TEST_APPID = 264710
probe = SESSION.get(f"https://steamcharts.com/app/{TEST_APPID}", timeout=20)
print(f"Status: {probe.status_code}  |  {len(probe.content):,} bytes")

soup = BeautifulSoup(probe.text, "lxml")
table = soup.find("table", class_="common-table")
print(f"table.common-table found: {table is not None}")

if table:
    rows = table.find_all("tr")
    print(f"Rows in table: {len(rows)}")
    print("\nFirst 5 data rows:")
    for row in rows[1:6]:   # skip header
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        print(f"  {cells}")

Status: 200  |  44,220 bytes
table.common-table found: True
Rows in table: 146

First 5 data rows:
  ['Last 30 Days', '3898.93', '+66.7', '+1.74%', '7233']
  ['February 2026', '3832.28', '-1534.71', '-28.60%', '7233']
  ['January 2026', '5366.99', '961.05', '+21.81%', '11997']
  ['December 2025', '4405.94', '1442.32', '+48.67%', '10468']
  ['November 2025', '2963.62', '-298.15', '-9.14%', '5547']


## 1. Build the scrape list from `steam_matches_manual.json`

In [3]:
with open(MATCHES_FILE, encoding="utf-8") as f:
    matches = json.load(f)

# Only keep entries that are a list with exactly one [appid, name] pair
scrape_list: list[dict] = []
skipped = []

for epic_name, value in matches.items():
    if not isinstance(value, list) or len(value) != 1 or not isinstance(value[0], list):
        skipped.append((epic_name, value if isinstance(value, str) else f"{len(value) if isinstance(value, list) else '?'} entries"))
        continue
    appid, steam_name = value[0]
    scrape_list.append({"epic_name": epic_name, "appid": int(appid), "steam_name": steam_name})

# Deduplicate by appid (same game given away multiple times)
seen_appids: set[int] = set()
unique_scrape: list[dict] = []
for entry in scrape_list:
    if entry["appid"] not in seen_appids:
        seen_appids.add(entry["appid"])
        unique_scrape.append(entry)

already_cached = {int(p.stem) for p in CACHE_DIR.glob("*.json")}
to_fetch = [e for e in unique_scrape if e["appid"] not in already_cached]

print(f"Total valid entries   : {len(scrape_list)}")
print(f"Unique app IDs        : {len(unique_scrape)}")
print(f"Already cached        : {len(already_cached)}")
print(f"To fetch              : {len(to_fetch)}  (~{len(to_fetch)*RATE_LIMIT_S/60:.1f} min)")
print(f"Skipped (no/multi/str): {len(skipped)}")
if skipped:
    for name, reason in skipped:
        print(f"  - {name!r}: {reason}")

Total valid entries   : 463
Unique app IDs        : 459
Already cached        : 445
To fetch              : 14  (~0.2 min)
Skipped (no/multi/str): 8
  - '3 out of 10: Season One (Ep 1)': Epic Games Store exclusive
  - '3 out of 10: Season Two': Epic Games Store exclusive
  - 'KID A MNESIA EXHIBITION': Epic Games Store exclusive
  - 'Antstream – Epic Welcome Pack': Epic Games Store exclusive
  - 'Rumbleverse Boom Boxer Pack (DLC)': Epic Games Store exclusive
  - 'Out of Bandwidth (placeholder small title slot)*': Placeholder
  - 'Additional minor DLC / cosmetics slots (holiday pack group)*': Placeholder
  - 'REDACTED (mystery day 2024)*': Placeholder


## 2. Scrape SteamCharts

In [4]:
import calendar

# Month name → zero-padded number, e.g. "January" → "01"
_MONTH_MAP = {name: f"{i:02d}" for i, name in enumerate(calendar.month_name) if i}


def parse_number(text: str) -> float | None:
    """'5,367.0' → 5367.0;  '11,997' → 11997.0;  '-' or '' → None"""
    text = text.replace(",", "").strip()
    if not text or text == "-":
        return None
    try:
        return float(text)
    except ValueError:
        return None


def parse_month(text: str) -> str | None:
    """'January 2026' → '2026-01';  'Last 30 Days' → None"""
    parts = text.strip().split()
    if len(parts) == 2 and parts[0] in _MONTH_MAP:
        return f"{parts[1]}-{_MONTH_MAP[parts[0]]}"
    return None


def fetch_monthly_table(appid: int) -> tuple[list[dict], str]:
    """
    Fetch the SteamCharts app page for *appid* and parse the server-rendered
    table.common-table.  Returns:
      records  — list of {month, avg_players, peak_players}
      status   — 'ok:N', '404', 'no_table', 'http:NNN', 'error:...'

    The table is fully server-side rendered; no JS execution needed.
    Columns: Month | Avg. Players | Gain | % Gain | Peak Players
    """
    url = f"https://steamcharts.com/app/{appid}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = SESSION.get(url, timeout=20)
            if resp.status_code == 404:
                return [], "404"
            if resp.status_code != 200:
                if attempt == MAX_RETRIES:
                    return [], f"http:{resp.status_code}"
                time.sleep(RATE_LIMIT_S * attempt)
                continue
            html = resp.text
            break
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                return [], f"error:{e}"
            time.sleep(RATE_LIMIT_S * attempt)

    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table", class_="common-table")
    if table is None:
        return [], "no_table"

    records = []
    for row in table.find_all("tr")[1:]:   # skip header row
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cells) < 5:
            continue
        month = parse_month(cells[0])
        if month is None:
            continue   # skip "Last 30 Days" rolling row
        avg  = parse_number(cells[1])
        peak = parse_number(cells[4])
        if avg is None:
            continue
        records.append({
            "month":        month,
            "avg_players":  avg,
            "peak_players": int(peak) if peak is not None else None,
        })

    return records, f"ok:{len(records)}"


print(f"Starting scrape of {len(to_fetch)} games (~{len(to_fetch)*RATE_LIMIT_S/60:.1f} min)...\n")
errors = []

for i, entry in enumerate(to_fetch):
    appid      = entry["appid"]
    epic_name  = entry["epic_name"]
    steam_name = entry["steam_name"]
    cache_path = CACHE_DIR / f"{appid}.json"

    monthly, status = fetch_monthly_table(appid)

    if not status.startswith("error:"):
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump({
                "appid":      appid,
                "steam_name": steam_name,
                "epic_name":  epic_name,
                "months":     monthly,
            }, f, indent=2)
    else:
        errors.append((appid, epic_name, status))

    print(f"[{i+1:>3}/{len(to_fetch)}] {appid:>8}  {status:<14}  {epic_name}")
    time.sleep(RATE_LIMIT_S)

print(f"\nDone. {len(errors)} network errors.")
if errors:
    for appid, name, err in errors:
        print(f"  {appid} {name!r}: {err}")

Starting scrape of 14 games (~0.2 min)...

[  1/14]   629090  http:500        Horace
[  2/14]   667810  http:500        Next Up Hero
[  3/14]   437160  http:500        The Lion’s Song
[  4/14]   595790  http:500        Hell is Other Demons
[  5/14]   468670  http:500        Speed Brawl
[  6/14]  1243690  http:500        Gods Will Fall
[  7/14]  1714180  http:500        Black Widow: Recharged
[  8/14]   916350  http:500        Saturnalia
[  9/14]   505740  ok:63           Eximius: Seize the Frontline
[ 10/14]  2845610  http:500        Dodo Peak
[ 11/14]  1096690  http:500        Mighty Fight Federation
[ 12/14]  1398910  http:500        Jorel’s Brother and The Most Important Game of the Galaxy
[ 13/14]  2439780  http:500        Godzilla Voxel Wars
[ 14/14]   782940  http:500        Rawmen: Food Fighter Arena

Done. 0 network errors.


## 3. Inspect empty results


In [5]:
# Find all cached files with an empty months list
empty = []
for p in sorted(CACHE_DIR.glob("*.json")):
    data = json.loads(p.read_text())
    if not data["months"]:
        empty.append((data["appid"], data["epic_name"], data["steam_name"]))

if empty:
    print(f"{len(empty)} game(s) returned no monthly data:\n")
    print(f"{'appid':>8}  {'epic_name':<45}  steam_name")
    print("-" * 80)
    for appid, epic, steam in empty:
        print(f"{appid:>8}  {epic:<45}  {steam}")
else:
    print("All cached games have at least one month of data.")


def delete_empty_caches(dry_run: bool = True) -> None:
    """Delete cache files with no monthly data so they are retried on next scrape.

    Call with dry_run=False to actually delete.
    Re-run cells 5 and 7 afterwards to re-scrape the removed games.
    """
    deleted = 0
    for p in sorted(CACHE_DIR.glob("*.json")):
        data = json.loads(p.read_text())
        if not data["months"]:
            if dry_run:
                print(f"[dry run] would delete {p.name}  ({data['epic_name']})")
            else:
                p.unlink()
                print(f"Deleted {p.name}  ({data['epic_name']})")
            deleted += 1
    if deleted == 0:
        print("Nothing to delete.")
    elif dry_run:
        print(f"\n{deleted} file(s) would be deleted. Call delete_empty_caches(dry_run=False) to confirm.")
    else:
        print(f"\nDeleted {deleted} file(s). Re-run cells 5 and 7 to retry.")

# Preview what would be deleted:
delete_empty_caches(dry_run=False)

13 game(s) returned no monthly data:

   appid  epic_name                                      steam_name
--------------------------------------------------------------------------------
 1096690  Mighty Fight Federation                        Mighty Fight Federation
 1243690  Gods Will Fall                                 Gods Will Fall
 1398910  Jorel’s Brother and The Most Important Game of the Galaxy  Jorel’s Brother and The Most Important Game of the Galaxy - Complete Edition
 1714180  Black Widow: Recharged                         Black Widow: Recharged
 2439780  Godzilla Voxel Wars                            Godzilla Voxel Wars
 2845610  Dodo Peak                                      Dodo Peak
  437160  The Lion’s Song                                The Lion's Song
  468670  Speed Brawl                                    Speed Brawl
  595790  Hell is Other Demons                           Hell is Other Demons
  629090  Horace                                         Horace
  6678

## 4. Build combined long-format CSV


In [6]:
# Load epic giveaway dates to attach treatment info
epic_df = pd.read_csv(DATA_DIR / "epic games data.csv")
epic_df["Start"] = pd.to_datetime(epic_df["Start"], dayfirst=True)
epic_df["End"]   = pd.to_datetime(epic_df["End"],   dayfirst=True)

# Build epic_name → earliest giveaway start (for games given away multiple times)
import re as _re
def clean_name(n):
    return _re.sub(r'\s*\(repeat[^)]*\)\s*$', '', str(n), flags=_re.IGNORECASE).strip()

epic_df["clean_name"] = epic_df["Game"].apply(clean_name)
first_giveaway = (
    epic_df.groupby("clean_name")["Start"]
    .min()
    .reset_index()
    .rename(columns={"clean_name": "epic_name", "Start": "first_giveaway_date"})
)

# Combine all cached JSON files into one DataFrame
rows = []
for cache_file in sorted(CACHE_DIR.glob("*.json")):
    with open(cache_file, encoding="utf-8") as f:
        data = json.load(f)
    for rec in data["months"]:
        rows.append({
            "appid":        data["appid"],
            "steam_name":   data["steam_name"],
            "epic_name":    data["epic_name"],
            "month":        rec["month"],
            "avg_players":  rec["avg_players"],
            "peak_players": rec["peak_players"],
        })

df = pd.DataFrame(rows)
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")

# Merge giveaway date
df = df.merge(first_giveaway, on="epic_name", how="left")

df = df.sort_values(["appid", "month"]).reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)

print(f"Saved {len(df):,} rows × {len(df.columns)} cols to {OUT_CSV}")
print(f"Games with data : {df['appid'].nunique()}")
print(f"Date range      : {df['month'].min().strftime('%Y-%m')} → {df['month'].max().strftime('%Y-%m')}")
df.head(10)

Saved 38,642 rows × 7 cols to ../raw_data/steamcharts/all_monthly.csv
Games with data : 446
Date range      : 2012-07 → 2026-02


,appid,steam_name,epic_name,month,avg_players,peak_players,first_giveaway_date
0,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-07-01,1.02,4,2021-10-14
1,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-08-01,0.41,4,2021-10-14
2,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-09-01,0.17,4,2021-10-14
3,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-10-01,0.29,3,2021-10-14
4,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-11-01,0.21,3,2021-10-14
5,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2012-12-01,0.23,3,2021-10-14
6,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2013-01-01,0.30,2,2021-10-14
7,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2013-02-01,0.32,4,2021-10-14
8,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2013-03-01,0.22,3,2021-10-14
9,7800,Stubbs the Zombie in Rebel Without a Pulse,Stubbs the Zombie in Rebel Without a Pulse,2013-04-01,0.16,3,2021-10-14


## 5. Quick sanity check


In [7]:
# Games with zero months of data (404 or no table)
cached_appids = {int(p.stem) for p in CACHE_DIR.glob("*.json")}
appids_with_data = set(df["appid"])
no_data_appids = cached_appids - appids_with_data

if no_data_appids:
    print(f"{len(no_data_appids)} app IDs returned no monthly data (delisted / private):")
    for aid in sorted(no_data_appids):
        cache_file = CACHE_DIR / f"{aid}.json"
        meta = json.loads(cache_file.read_text())
        print(f"  {aid:>8}  {meta['epic_name']}")
else:
    print("All cached games have at least one month of data.")

print()
print("Rows per game (sample):")
print(df.groupby("steam_name")["month"].count().sort_values().head(10).to_string())

All cached games have at least one month of data.

Rows per game (sample):
steam_name
Spelldrifter                       1
The Forest Quartet                 2
Wildcat Gun Machine                2
LumbearJack                        3
Out of Line                        3
Daemon X Machina: Titanic Scion    5
Against All Odds                   6
Jitsu Squad                        7
Universe For Sale                  8
CYGNI: All Guns Blazing            9


## 6. Treatment variable construction

Adds two columns to `all_monthly.csv` for use in the DiD specifications:

- **`all_giveaway_dates`** — semicolon-separated list of every Epic giveaway start date for this game (sorted).
- **`repeat_event_it`** — 1 if month *t* falls within `[0, 12]` months after a *non-first* giveaway; used in the separated spec to control for contamination of the first-giveaway post-window.


In [8]:
# Reload panel so this cell is idempotent
df = pd.read_csv(OUT_CSV, parse_dates=["month", "first_giveaway_date"])

# --- all_giveaway_dates ---
# For each unique epic_name, collect every giveaway start date (sorted).
epic_df_fresh = pd.read_csv(DATA_DIR / "epic games data.csv")
epic_df_fresh["Start"] = pd.to_datetime(epic_df_fresh["Start"], dayfirst=True)
epic_df_fresh["clean_name"] = epic_df_fresh["Game"].apply(clean_name)

all_dates_map = (
    epic_df_fresh.groupby("clean_name")["Start"]
    .apply(lambda x: sorted(x.tolist()))
    .to_dict()
)  # epic_name -> [Timestamp, ...]

all_dates_df = pd.DataFrame([
    {"epic_name": name,
     "all_giveaway_dates": ";".join(d.strftime("%Y-%m-%d") for d in dates)}
    for name, dates in all_dates_map.items()
])

df = df.drop(columns=["all_giveaway_dates"], errors="ignore")
df = df.merge(all_dates_df, on="epic_name", how="left")

# --- repeat_event_it ---
# 1 if month t falls in [0, 12] months after any giveaway that is NOT the first.
def compute_repeat_event(month, dates_str):
    if not isinstance(dates_str, str):
        return 0
    dates = [pd.Timestamp(d) for d in dates_str.split(";")]
    if len(dates) <= 1:
        return 0
    for repeat_date in dates[1:]:   # skip first giveaway
        months_since = (month.year  - repeat_date.year)  * 12 + \
                       (month.month - repeat_date.month)
        if 0 <= months_since <= 12:
            return 1
    return 0

df["repeat_event_it"] = df.apply(
    lambda row: compute_repeat_event(row["month"], row["all_giveaway_dates"]),
    axis=1,
)

df = df.sort_values(["appid", "month"]).reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)

print(f"Saved {len(df):,} rows × {len(df.columns)} cols to {OUT_CSV}")
print(f"repeat_event_it = 1 : {df['repeat_event_it'].sum():,} game-months")
print(f"Games with ≥1 contaminated month : {df[df['repeat_event_it']==1]['appid'].nunique()}")

# --- Spot-check: A Plague Tale: Innocence ---
# First giveaway: Aug 2021 (k=0).  2nd: Dec 2023 (k=28),  3rd: Jan 2024 (k=29).
# repeat_event_it should be 1 from Dec 2023 through Jan 2025, 0 elsewhere.
plague = df[df["epic_name"].str.contains("Plague Tale", na=False)].copy()
plague["k"] = ((plague["month"].dt.year  - plague["first_giveaway_date"].dt.year) * 12 +
               (plague["month"].dt.month - plague["first_giveaway_date"].dt.month))
print("\nA Plague Tale: Innocence — window around 2nd/3rd giveaways (k=25–43):")
print(plague[["month", "k", "repeat_event_it"]]
      .query("25 <= k <= 43")
      .to_string(index=False))

Saved 38,642 rows × 9 cols to ../raw_data/steamcharts/all_monthly.csv
repeat_event_it = 1 : 838 game-months
Games with ≥1 contaminated month : 63

A Plague Tale: Innocence — window around 2nd/3rd giveaways (k=25–43):
     month  k  repeat_event_it
2023-09-01 25                0
2023-10-01 26                0
2023-11-01 27                0
2023-12-01 28                1
2024-01-01 29                1
2024-02-01 30                1
2024-03-01 31                1
2024-04-01 32                1
2024-05-01 33                1
2024-06-01 34                1
2024-07-01 35                1
2024-08-01 36                1
2024-09-01 37                1
2024-10-01 38                1
2024-11-01 39                1
2024-12-01 40                1
2025-01-01 41                1
2025-02-01 42                0
2025-03-01 43                0
